# Importing all Libreries

In [ ]:
import os
import torch
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModel

from huggingface_hub import login
from google.colab import userdata


# 1. Device Allocation & Runtime Verification

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Active Compute Device: {device}")
if torch.cuda.is_available():
    print(f"GPU Model: {torch.cuda.get_device_name(0)}")

Active Compute Device: cuda
GPU Model: Tesla T4


HF_TOKEN (secret key)

In [ ]:
userdata.get('HF_TOKEN')
login(userdata.get("HF_TOKEN"))

# 2. Load Model & Tokenizer

In [ ]:
MODEL_NAME = "mental/mental-bert-base-uncased"
print(f"Loading pre-trained weights for: {MODEL_NAME}")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModel.from_pretrained(MODEL_NAME)

Loading pre-trained weights for: mental/mental-bert-base-uncased


pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  438MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: mental/mental-bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
pooler.dense.bias                          | MISSING    | 
pooler.dense.weight                        | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
model = model.to(device)
model.eval()

# Explicitly freeze all parameters
for param in model.parameters():
    param.requires_grad = False

print("MentalBERT parameters successfully frozen.")

MentalBERT parameters successfully frozen.


# 3. Load Master Dataset

In [ ]:
CSV_PATH = "/content/drive/MyDrive/Thesis/Dataset/results_master_stacked.csv"
df = pd.read_csv(CSV_PATH)
print(f"Loaded master dataset with {len(df)} rows.")

model.safetensors: reconstructing file:   0%|          |  0.00B /  438MB            

model.safetensors: downloading bytes:           |  0.00B            

Loaded master dataset with 139 rows.


# 4. Long-Document Feature Extraction Engine

In [ ]:
def extract_mentalbert_embedding(
    text: str,
    max_length: int = 512,
    stride: int = 256,
    batch_size: int = 8
) -> np.ndarray:
    """
    Extracts a robust 768-dimensional representation from arbitrary-length transcripts
    using sliding-window tokenization and attention-weighted mean pooling across chunks.
    """
    if not isinstance(text, str) or not text.strip():

        return np.zeros(768, dtype=np.float32)
    encoded = tokenizer(
        text,
        max_length=max_length,
        truncation=True,
        stride=stride,
        return_overflowing_tokens=True,
        return_offsets_mapping=False,
        padding="max_length",
        return_tensors="pt"
    )

    input_ids = encoded["input_ids"]
    attention_mask = encoded["attention_mask"]
    num_chunks = input_ids.size(0)

    chunk_embeddings = []
    with torch.inference_mode():
        for i in range(0, num_chunks, batch_size):
            b_input_ids = input_ids[i : i + batch_size].to(device)
            b_attn_mask = attention_mask[i : i + batch_size].to(device)

            outputs = model(input_ids=b_input_ids, attention_mask=b_attn_mask)
            token_embeddings = outputs.last_hidden_state
            expanded_mask = b_attn_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
            sum_embeddings = torch.sum(token_embeddings * expanded_mask, dim=1)
            sum_mask = torch.clamp(expanded_mask.sum(dim=1), min=1e-9)
            mean_pooled = sum_embeddings / sum_mask

            chunk_embeddings.append(mean_pooled.cpu().numpy())
    all_chunks = np.vstack(chunk_embeddings)
    transcript_embedding = np.mean(all_chunks, axis=0)

    return transcript_embedding.astype(np.float32)

# 5. Execute Extraction Loop

In [ ]:
embeddings_list = []
print("Extracting Layer-1 text embeddings across transcripts...")

for idx, text in enumerate(tqdm(df["clean_transcript"], desc="MentalBERT Extraction")):
    emb = extract_mentalbert_embedding(text)
    embeddings_list.append(emb)

text_features_matrix = np.vstack(embeddings_list)
print(f"Output Text Feature Matrix Shape: {text_features_matrix.shape}")

Extracting Layer-1 text embeddings across transcripts...


MentalBERT Extraction:   0%|          | 0/139 [00:00<?, ?it/s]

Output Text Feature Matrix Shape: (139, 768)


# 6. Serialization

In [ ]:
OUTPUT_DIR = "/content/drive/MyDrive/Thesis/Dataset/"
np.save(
    f"{OUTPUT_DIR}mentalbert_features_768d.npy", text_features_matrix
)
emb_columns = [f"text_emb_{i:03d}" for i in range(768)]
df_embeddings = pd.DataFrame(text_features_matrix, columns=emb_columns)

df_out = pd.concat([df.reset_index(drop=True), df_embeddings], axis=1)
df_out.to_csv(
    f"{OUTPUT_DIR}results_master_with_text_features.csv", index=False
)
print(f"Output files securely preserved in Google Drive: {OUTPUT_DIR}")

Output files securely preserved in Google Drive: /content/drive/MyDrive/Thesis/Dataset/


## Summary of the Pipeline

This notebook successfully loaded a pre-trained MentalBERT model and tokenizer, extracted 768-dimensional embeddings from a master dataset's `clean_transcript` column using a sliding-window approach with attention-weighted mean pooling, and then serialized these embeddings to both a `.npy` file and a CSV file, augmenting the original dataset with the new text features. All output files are securely stored in Google Drive.